Imports & Setups

In [ ]:
import os
import yaml
from uuid import uuid1
from pathlib import Path
from dotenv import load_dotenv
from langsmith import traceable
from tools import available_tools
from rich.markdown import Markdown
from langgraph_utils import console
from typing import TypedDict, Annotated, List
from langgraph.types import interrupt, Command
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.runnables import RunnableConfig
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.prebuilt import ToolNode, tools_condition
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import BaseMessage, SystemMessage, HumanMessage, AIMessage

load_dotenv()
PARAMS_CONFIGS = yaml.safe_load(Path("configs/params.yaml").read_text())
PROMPTS_CONFIGS = yaml.safe_load(Path("configs/prompts.yaml").read_text())

os.environ["LANGSMITH_PROJECT"] = PARAMS_CONFIGS["LANGSMITH_PROJECT"]
os.environ["LANGSMITH_TRACING_V2"] = str(PARAMS_CONFIGS["LANGCHAIN_TRACING_V2"]).lower()


Model

In [ ]:
model = ChatGoogleGenerativeAI(**PARAMS_CONFIGS["llm"])
bound_model = model.bind_tools(available_tools)

State

In [ ]:
class ToolState(TypedDict):
    messages: Annotated[List, add_messages]


Node & Utility Functions

In [ ]:
@traceable(name="set_system_instructions")
def set_system_instructions(state: ToolState):
    if any(isinstance(message, SystemMessage) for message in state["messages"]):
        return {}

    return {"messages": [SystemMessage(content=PROMPTS_CONFIGS["chat"]["system"])]}


@traceable(name="chat")
def chat(state: ToolState, config):
    messages = state["messages"]
    if not any(isinstance(message, SystemMessage) for message in messages):
        messages = [
            SystemMessage(content=PROMPTS_CONFIGS["chat"]["system"]),
            *messages,
        ]
    res = bound_model.invoke(input=state["messages"], config=config)
    return {"messages": [res]}


@traceable(name="approve_tool")
def approve_tools(state: ToolState):
    required_tools = [tool_call["name"] for tool_call in state["messages"][-1].tool_calls]

    approved = interrupt(
        {
            "type": "approval",
            "reason": "The model requires tool(s) approval.",
            "required_tools": required_tools,
        }
    )
    if approved:
        return Command(goto="tools")

    return Command(goto=END)


tools = ToolNode(available_tools)


Init Graph

In [ ]:
graph = StateGraph(ToolState)

Add Nodes

In [ ]:
graph.add_node("set_system_instructions", set_system_instructions)
graph.add_node("chat", chat)
graph.add_node("approve_tools", approve_tools)
graph.add_node("tools", tools)


Add Edges

In [ ]:
graph.add_edge(START, "set_system_instructions")
graph.add_edge("set_system_instructions", "chat")
graph.add_conditional_edges("chat", tools_condition, {END: END, "tools": "approve_tools"})

graph.add_edge("approve_tools", "tools")
graph.add_edge("tools", "chat")


Compilation

In [ ]:
checkpointer = InMemorySaver()
workflow = graph.compile(checkpointer=checkpointer)
workflow

Config

In [ ]:
thread_id = str(uuid1())
config = RunnableConfig(
    configurable={
        "thread_id": thread_id,
    },
    metadata={
        "thread_id": thread_id,
        "environment": os.getenv("APP_ENV", "default"),
        "app": "chatbot_with_hitl",
    },
    run_name="hitl_turn",
)
console.print(config)

Execution

In [ ]:
def invoke_with_hitl(input_, config):
    state = workflow.invoke(input_, config=config)

    while "__interrupt__" in state:
        interrupt = state["__interrupt__"][0].value
        required_tools = ", ".join(interrupt["required_tools"])

        approved = (
            input(
                "\n"
                " Tool Approval Required\n"
                f"Reason:\n  {interrupt['reason']}\n\n"
                f"Requested tool(s):\n  • {required_tools}\n\n"
                "Approve execution? [y/n]: "
            )
            .strip()
            .lower()
            == "y"
        )

        state = workflow.invoke(
            Command(resume=approved),
            config=config,
        )

    return state


In [11]:
user_query = "Convert 500 GBP to INR using the latest exchange rate. Find the current prices of both the Apple Polishing Cloth and the Apple USB-C Charge Cable (1 m) in India. Then calculate how many of each I could buy with the converted amount and tell me which gives me the higher quantity."
final_state = invoke_with_hitl(
    input_={"messages": [HumanMessage(content=user_query)]},
    config=config,
)

: 

: 

In [ ]:
console.print(final_state["messages"])


In [ ]:
console.print(Markdown(final_state["messages"][-1].content[0]["text"]))
